# N17 · Tool-calling Agent

- 理论：`08-V1-1/01-工具调用与agent.md` §1–§3
- 源码：`v1-1-agentic-tool/driver/{evidence_loop.py,v1_contract.py,v1_transports.py}`、`gpu/gpu_runtime/{server.py,actor.py,runtime.py}`
- 代码：`08-V1-1/code/tool_agent.py`
- Colab：纯 Python，无需依赖。

## 理论回顾

agent 循环 = 生成 -> 解析工具调用（OpenAI `tool_calls` 或 DSL `{"calls":[...]}`）-> 执行 -> 回灌 -> 最终答案。
证据环：`Evidence` 聚合成 $\phi=\tanh(\sum w)$，把树外证据以贝叶斯方式注入先验。

In [1]:
import json, math, re
def evidence_composite(evs): return math.tanh(sum(e['weight'] for e in evs))
LEAN_DB = {'sq_connected': 'lemma sq_connected {a b : R} : a^2=b^2 <-> a=b or a=-b',
           'add_comm': 'lemma add_comm (a b : N) : a+b=b+a'}
FILE_DB = {'Mathlib/Algebra/Ring/Basic.lean': 'import Mathlib.Algebra.Ring.Basic'}
def run_tool(name, args):
    if name == 'leanSearch':
        return {'results': [v for k,v in LEAN_DB.items() if args.get('query','') in k] or list(LEAN_DB.values())[:1]}
    if name == 'fileSearch':
        return {'content': FILE_DB.get(args.get('path',''), '(not found)')}
    if name == 'calculator':
        return {'value': eval(re.sub(r'[^0-9+\-*/(). ]','',args.get('expr','0')), {'__builtins__':{}})}
    return {'error': 'unknown tool ' + name}
print('tools ready')

tools ready


In [2]:
def extract_json(text):
    start = text.find('{')
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc: esc = False
                elif ch == '\\': esc = True
                elif ch == '"': in_str = False
                continue
            if ch == '"': in_str = True
            elif ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try: return json.loads(text[start:i+1])
                    except json.JSONDecodeError: break
        start = text.find('{', start+1)
    return None
def parse_calls(text):
    obj = extract_json(text)
    if not isinstance(obj, dict): return []
    out = []
    for c in obj.get('calls', []) or []:
        if isinstance(c, dict) and 'name' in c: out.append({'name': c['name'], 'args': c.get('args', {})})
    for tc in obj.get('tool_calls', []) or []:
        fn = tc.get('function', {}); args = fn.get('arguments', {})
        if isinstance(args, str):
            try: args = json.loads(args)
            except Exception: args = {}
        out.append({'name': fn.get('name',''), 'args': args})
    return out
print(parse_calls('{"calls":[{"name":"leanSearch","args":{"query":"sq_connected"}}]}'))
print(parse_calls('{"tool_calls":[{"function":{"name":"fileSearch","arguments":"{\\"path\\":\\"x\\"}"}}]}'))

[{'name': 'leanSearch', 'args': {'query': 'sq_connected'}}]
[{'name': 'fileSearch', 'args': {'path': 'x'}}]


In [3]:
def agent_loop(answers, max_steps=4):
    trajectory = []
    for step in range(max_steps):
        msg = answers[step] if step < len(answers) else 'final: done'
        calls = parse_calls(msg)
        if not calls:
            return msg, trajectory
        for c in calls:
            res = run_tool(c['name'], c['args'])
            trajectory.append((c['name'], res))
    return 'max_steps', trajectory
plan = ['{"calls":[{"name":"leanSearch","args":{"query":"sq_connected"}}]}',
        '{"calls":[{"name":"calculator","args":{"expr":"2+3"}}]}',
        'final: use sq_connected then linarith']
answer, traj = agent_loop(plan)
for name, res in traj: print(name, '->', str(res)[:70])
print('FINAL:', answer)
evs = [{'kind':'leanSearch','weight':0.25}, {'kind':'toolCall','weight':0.25}]
print('phi = tanh(sum w) =', round(evidence_composite(evs), 4))

leanSearch -> {'results': ['lemma sq_connected {a b : R} : a^2=b^2 <-> a=b or a=-b']
calculator -> {'value': 5}
FINAL: final: use sq_connected then linarith
phi = tanh(sum w) = 0.4621


In [4]:
# 证据注入 A/B：无证据 vs 有证据（这里用启发式打分模拟 value 变化）
def value_estimate(state, evidence):
    base = -2.0
    bonus = 1.0 if any(e['kind'] == 'leanSearch' for e in evidence) else 0.0
    return base + bonus
states = ['a^2+1=b^2+1 -> a=b or a=-b']
for s in states:
    v0 = value_estimate(s, [])
    v1 = value_estimate(s, [{'kind':'leanSearch','weight':0.25}])
    print(f'state={s!r} value0={v0:.2f} value1={v1:.2f} delta={v1-v0:+.2f}')

state='a^2+1=b^2+1 -> a=b or a=-b' value0=-2.00 value1=-1.00 delta=+1.00


## 思考题
1. 为什么 `phi=tanh(sum w)` 要饱和压缩？改成线性会怎样？
2. 找第一个平衡 `{...}` 的解析在字符串含花括号、多个 JSON、输出截断时会怎样？
3. 证据注入后如何判断它真的“起作用”？（提示：看 value delta 与后续 tactic 命中率）
4. 把本地工具换成 opencode 的真实搜索，需要改哪一层？（transport / evidence producer）

## 预期输出
- `parse_calls` 能同时解析 DSL 与 OpenAI 两种格式；
- 轨迹打印 leanSearch/calculator 的结果；`FINAL:` 为最终答案；
- `phi=tanh(0.5)=0.4621`；证据注入后 `delta=+1.00`。